# Nädal 7 – Grupitöö

## Roll C – RFM kliendisegmenteerimine

Minu ülesanne on arvutada Roll B puhastatud andmete põhjal iga kliendi Recency, Frequency ja Monetary väärtused ning jagada kliendid RFM-segmentidesse.

In [1]:
import pandas as pd

df = pd.read_csv("df_cleaned_roll_B.csv")

print("DataFrame mõõtmed:", df.shape)
print("\nVeerud:")
print(df.columns.tolist())

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'df_cleaned_roll_B.csv'

In [2]:
import pandas as pd

df = pd.read_csv("df_cleaned_roll_B (2).csv")

print("DataFrame mõõtmed:", df.shape)
print("\nVeerud:")
print(df.columns.tolist())

df.head()

DataFrame mõõtmed: (8621, 19)

Veerud:
['sale_id', 'invoice_id', 'sale_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'total_price', 'channel', 'store_location', 'payment_method', 'first_name', 'last_name', 'email', 'phone', 'city', 'registration_date', 'loyalty_tier', 'birth_year']


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0


In [3]:
df["sale_date"] = pd.to_datetime(
    df["sale_date"],
    errors="coerce"
)

print("Puuduvad väärtused:")
print(df[["customer_id", "sale_date", "total_price"]].isnull().sum())

print("\nNull- või negatiivsed tehingud:")
print((df["total_price"] <= 0).sum())

print("\nDuplikaatsed invoice_id-d:")
print(df.duplicated(subset="invoice_id").sum())

print("\nKuupäevavahemik:")
print(df["sale_date"].min(), "kuni", df["sale_date"].max())

print("\nUnikaalseid kliente:")
print(df["customer_id"].nunique())

Puuduvad väärtused:
customer_id    0
sale_date      0
total_price    0
dtype: int64

Null- või negatiivsed tehingud:
0

Duplikaatsed invoice_id-d:
0

Kuupäevavahemik:
2023-01-01 00:00:00 kuni 2026-06-28 00:00:00

Unikaalseid kliente:
2524


In [4]:
analysis_date = df["sale_date"].max() + pd.Timedelta(days=1)

rfm = (
    df.groupby("customer_id")
    .agg(
        Recency=("sale_date", lambda x: (analysis_date - x.max()).days),
        Frequency=("sale_id", "count"),
        Monetary=("total_price", "sum")
    )
    .reset_index()
)

print("Analüüsi kuupäev:", analysis_date.date())
print("Klientide arv:", rfm.shape[0])

rfm.head()

Analüüsi kuupäev: 2026-06-29
Klientide arv: 2524


,customer_id,Recency,Frequency,Monetary
0,2001.0,577,2,203.92
1,2004.0,557,2,1198.56
2,2005.0,634,4,959.60
3,2006.0,963,1,327.06
4,2007.0,515,1,318.63


In [5]:
rfm["R_score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["F_score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_score"] = pd.qcut(
    rfm["Monetary"],
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["RFM_Score"] = (
    rfm["R_score"]
    + rfm["F_score"]
    + rfm["M_score"]
)

print(rfm[[
    "customer_id",
    "Recency",
    "Frequency",
    "Monetary",
    "R_score",
    "F_score",
    "M_score",
    "RFM_Score"
]].head())

print("\nSkooride vahemikud:")
print(
    rfm[["R_score", "F_score", "M_score", "RFM_Score"]]
    .agg(["min", "max"])
)

   customer_id  Recency  Frequency  Monetary  R_score  F_score  M_score  \
0       2001.0      577          2    203.92        4        1        1   
1       2004.0      557          2   1198.56        4        1        5   
2       2005.0      634          4    959.60        3        4        4   
3       2006.0      963          1    327.06        1        1        2   
4       2007.0      515          1    318.63        5        1        2   

   RFM_Score  
0          6  
1         10  
2         11  
3          4  
4          8  

Skooride vahemikud:
     R_score  F_score  M_score  RFM_Score
min        1        1        1          3
max        5        5        5         15


In [6]:
def assign_segment(score):
    if score >= 13:
        return "VIP Champions"
    elif score >= 10:
        return "Loyal Customers"
    elif score >= 7:
        return "Potential Loyalists"
    elif score >= 4:
        return "At Risk"
    else:
        return "Lost"


rfm["Segment"] = rfm["RFM_Score"].apply(assign_segment)

print("Segmentide jaotus:")
print(rfm["Segment"].value_counts())

print("\nKontroll – puuduvad segmendid:")
print(rfm["Segment"].isnull().sum())

Segmentide jaotus:
Segment
Potential Loyalists    720
Loyal Customers        678
At Risk                541
VIP Champions          464
Lost                   121
Name: count, dtype: int64

Kontroll – puuduvad segmendid:
0


In [7]:
segment_summary = (
    rfm
    .groupby("Segment")
    .agg(
        Klientide_arv=("customer_id", "count"),
        Keskmine_Recency=("Recency", "mean"),
        Keskmine_Frequency=("Frequency", "mean"),
        Keskmine_Monetary=("Monetary", "mean"),
        Kogukaive=("Monetary", "sum")
    )
    .reset_index()
)

segment_summary["Klientide_osakaal"] = (
    segment_summary["Klientide_arv"]
    / rfm.shape[0]
    * 100
)

segment_summary["Kaibe_osakaal"] = (
    segment_summary["Kogukaive"]
    / segment_summary["Kogukaive"].sum()
    * 100
)

print("Segmentide kokkuvõte:")
print(segment_summary.round(2))

print("\nKontroll – kliente kokku:")
print(segment_summary["Klientide_arv"].sum())

print("\nKontroll – analüüsitud kogukäive:")
print(round(segment_summary["Kogukaive"].sum(), 2))

Segmentide kokkuvõte:
               Segment  Klientide_arv  Keskmine_Recency  Keskmine_Frequency  \
0              At Risk            541            806.41                1.56   
1                 Lost            121           1008.36                1.00   
2      Loyal Customers            678            641.23                3.74   
3  Potential Loyalists            720            688.51                2.36   
4        VIP Champions            464            536.90                7.37   

   Keskmine_Monetary  Kogukaive  Klientide_osakaal  Kaibe_osakaal  
0             324.55  175581.26              21.43           8.12  
1             162.13   19617.29               4.79           0.91  
2             950.78  644630.52              26.86          29.82  
3             571.48  411468.25              28.53          19.04  
4            1961.41  910096.47              18.38          42.11  

Kontroll – kliente kokku:
2524

Kontroll – analüüsitud kogukäive:
2161393.79


In [8]:
rfm.to_csv(
    "rfm_segments_roll_C.csv",
    index=False,
    encoding="utf-8-sig"
)

segment_summary.to_csv(
    "rfm_segment_summary_roll_C.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Failid salvestatud:")
print("- rfm_segments_roll_C.csv")
print("- rfm_segment_summary_roll_C.csv")

Failid salvestatud:
- rfm_segments_roll_C.csv
- rfm_segment_summary_roll_C.csv


## Roll C tulemused

RFM-analüüsi kaasati kokku **2524 klienti**.

Kliendid jagunesid segmentidesse järgmiselt:

- **VIP Champions:** 464 klienti
- **Loyal Customers:** 678 klienti
- **Potential Loyalists:** 720 klienti
- **At Risk:** 541 klienti
- **Lost:** 121 klienti

Kõige suurem segment oli **Potential Loyalists**, kuhu kuulus 720 klienti ehk 28,53% analüüsitud klientidest.

Kõige väärtuslikum segment oli **VIP Champions**. Sellesse kuulus 464 klienti ehk 18,38% klientidest, kuid segment andis 42,11% analüüsitud kogukäibest.

VIP Champions ja Loyal Customers andsid kokku **71,93% kogukäibest**, mistõttu on nende hoidmine ettevõtte jaoks kõige olulisem.

At Risk segmenti kuulus 541 klienti. Need kliendid vajavad tagasivõitmise kampaaniat, sest nende ostuaktiivsus on vähenenud.

Roll C väljundina valmisid:

- kliendipõhine RFM-tabel koos skooride ja segmentidega;
- segmentide kokkuvõttetabel;
- fail `rfm_segments_roll_C.csv`;
- fail `rfm_segment_summary_roll_C.csv`.